In [14]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pathlib
import shutil
import os
import requests
import time

curr_dir = pathlib.Path(".")
data_dir = curr_dir / "RTS-GMLC-master" / "RTS_Data"
solar_data_dir = curr_dir / "solar-data"

In [15]:
# Read in bus geodata and generation data for IEEE RTS GMLC (Reliability Test System)
df_bus = pd.read_csv(data_dir / "SourceData" / "bus.csv", index_col=[0])
df_geodata = df_bus[["lat", "lng"]]
df_gen_full = pd.read_csv(data_dir / "SourceData" / "gen.csv", index_col=[0])

In [16]:
# Get solar geodata
solar_geodata = df_geodata.loc[df_gen_full.loc[df_gen_full["Unit Type"].isin(["PV", "RTPV"]), "Bus ID"]].drop_duplicates()

In [ ]:
# Read in API key
with open("nrel-api-key.txt", "r") as f:
    API_KEY = f.readlines()[0].strip()

# Build request

url = (
    "https://developer.nrel.gov/api/nsrdb/v2/solar/nsrdb-GOES-aggregated-v4-0-0-download"
    ".csv"
    f"?api_key={API_KEY}"
)

for bus_id, row in solar_geodata.iterrows():

    print(f"Retrieving solar data for bus {bus_id}...")
    
    # Create directory for wind data for all years
    bus_solar_data_dir = solar_data_dir / f"bus{bus_id}"
    bus_solar_data_dir.mkdir(exist_ok=True, parents=True)
    
    # Extract bus geodata for request
    lat, lon = row.lat, row.lng
    point_wkt = f"POINT({lon} {lat})"
    
    for year in np.arange(1998, 2024):
        print(f"Year: {year}")
        payload = {
            "wkt": point_wkt,
            "names": str(year),
            "interval": 30,
            "utc": "true",
            "leap_day": "true",
            "attributes": "ghi,dni,dhi,air_temperature,wind_speed,relative_humidity,surface_albedo",
            "email": "charlesgulian@berkeley.edu",
            "full_name": "Charles Gulian",
            "affiliation": "UC Berkeley IEOR",
            "reason": "Research",
            "mailing_list": "false"
        }
        
        # Send request
        print("Submitting WTK data request to NREL...")
        response = requests.post(url, data=payload, timeout=60)
        
        if response.status_code == 200:
            print("Request submitted successfully.")

            # Save CSV file
            fname = bus_solar_data_dir / f"NSRDB_weather-inputs_bus{bus_id}_{year}.csv"
            with open(fname, "wb") as f:
                f.write(response.content)

            # Re-open CSV with pandas and clean
            df_solar = pd.read_csv(fname, skiprows=2) # Remove header
            df_solar.columns = [c.strip() for c in df_solar.columns] # Clean columns names
            df_solar["datetime"] = pd.to_datetime(df_solar[["Year", "Month", "Day", "Hour", "Minute"]]) # Create datetime column
            df_solar = df_solar[["datetime", "GHI", "DNI", "DHI", "Temperature", "Wind Speed", "Relative Humidity", "Surface Albedo"]].set_index("datetime") # Restrict to columns of interest + change index
            df_solar.index = pd.to_datetime(df_solar.index) # Convert to datetime index
            df_solar = df_solar.resample("h").asfreq() # Re-sample to hourly frequency

            # Save file again
            df_solar.to_csv(fname)

            del response
            del df_solar
            
        else:
            print(f"Request failed with status {response.status_code}")
            print(response.text)
            break
            
    time.sleep(2)
    print("Done.")

Retrieving solar data for bus 320...
Year: 1998
Submitting WTK data request to NREL...
Request submitted successfully.
Year: 1999
Submitting WTK data request to NREL...
